# Preliminary data processing

This script takes in the results from assembles-datasets and creates certain userful variables in the data.

In [1]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import matplotlib as mpl
import matplotlib.ticker as ticker

import numpy as np
import pandas as pd
import os
import plotly.graph_objects as go
import plotly.express as px
import pickle

# import clustering packages
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import seaborn as sns

plt.style.use('custom.mplstyle')
%config InlineBackend.figure_format = 'retina'
from tqdm import tqdm

from stoch_sim_model import *

## 0. Load data and build datasets

In [2]:
# Load data from infections
reg_model = ''
runs = '-5-'
comment = "acute_all-sparse-reg"

d_mean = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/stacked_data'+runs+'runs'+'-'+comment+'.pkl'
mean_df = pd.read_pickle(d_mean)

with pd.option_context('display.max_columns', None):
    display(mean_df)

,psi_Na_I,psi_Na_HI,psi_Na_HE,L0_Na,psi_NE_I,psi_NE_HI,psi_NE_HE,L0_NE,psi_EM_I,psi_EM_HI,psi_EM_HE,L0_EM,psi_EE_I,psi_EE_HI,psi_EE_HE,L0_EE,d_I,K_IE,b_I,K_EH,N_0,S_0,I_0,d_S,d_IE,b_H,d_H,max_Na,b_myc,d_myc,myc_thresh,t_bind,t_unbind,t_Na_div,t_E_div,t_M_div,t_E_die,t_act,p_load,T_max_pI,T_min_pI,harm_pI,harm_pS,max_pE,T_pE_max,T_pE_start,max_eM,T_eM_min,max_cM,T_cM_min,int_pHE,int_pHI,min_pS
0,-3.0,-3.0,-3.0,-5.0,0.0,0.0,0.0,-5.0,0.0,0.0,0.0,-5.0,0.0,0.0,0.0,-5.0,0.01,100000.000000,1.000000e-01,50000.0,100.000000,10000000.0,1000000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,5.513184e+07,20.0000,0.0,5.303137e+06,0.000000,198.0,20.0000,20.0000,0.0,0.000000,0.0,0.0,0.000000,2.073569e+06,6.027997e+06
1,-3.0,-3.0,-3.0,-5.0,0.0,0.0,0.0,-5.0,0.0,0.0,0.0,-5.0,0.0,0.0,0.0,-5.0,0.01,100000.000000,1.000000e-01,50000.0,234.034732,10000000.0,1000000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,5.513184e+07,20.0000,0.0,5.303137e+06,0.000000,468.0,20.0000,20.0000,0.0,0.000000,0.0,0.0,0.000000,2.073569e+06,6.027997e+06
2,-3.0,-3.0,-3.0,-5.0,0.0,0.0,0.0,-5.0,0.0,0.0,0.0,-5.0,0.0,0.0,0.0,-5.0,0.01,100000.000000,1.000000e-01,50000.0,547.722558,10000000.0,1000000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,5.513184e+07,20.0000,0.0,5.303137e+06,0.000000,1094.0,20.0000,20.0000,0.0,0.000000,0.0,0.0,0.000000,2.073569e+06,6.027997e+06
3,-3.0,-3.0,-3.0,-5.0,0.0,0.0,0.0,-5.0,0.0,0.0,0.0,-5.0,0.0,0.0,0.0,-5.0,0.01,100000.000000,1.000000e-01,50000.0,1281.861019,10000000.0,1000000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,5.513184e+07,20.0000,0.0,5.303137e+06,0.000000,2562.0,20.0000,20.0000,0.0,0.000000,0.0,0.0,0.000000,2.073569e+06,6.027997e+06
4,-3.0,-3.0,-3.0,-5.0,0.0,0.0,0.0,-5.0,0.0,0.0,0.0,-5.0,0.0,0.0,0.0,-5.0,0.01,100000.000000,1.000000e-01,50000.0,3000.000000,10000000.0,1000000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,5.513184e+07,20.0000,0.0,5.303137e+06,0.000000,6000.0,20.0000,20.0000,0.0,0.000000,0.0,0.0,0.000000,2.073569e+06,6.027997e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40352385,3.0,3.0,3.0,5.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,5.0,1.00,129154.966501,1.500000e-07,50000.0,1000.000000,10000000.0,1000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,1.104028e+06,20.0000,0.0,1.530945e+06,62578.029653,134864.6,13.1152,7.3136,118787.8,1.451626,3249.6,5.0,157265.891991,5.136801e+05,8.557431e+06
40352386,3.0,3.0,3.0,5.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,5.0,1.00,215443.469003,1.500000e-07,50000.0,1000.000000,10000000.0,1000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,1.985738e+06,20.0000,0.0,2.588873e+06,66576.161760,147996.2,13.3712,7.7472,130732.0,1.433143,3301.0,5.0,180670.409755,9.206810e+05,7.544888e+06
40352387,3.0,3.0,3.0,5.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,5.0,1.00,359381.366380,1.500000e-07,50000.0,1000.000000,10000000.0,1000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,2.951098e+06,19.5584,0.0,3.604477e+06,62188.886778,142508.0,13.4144,7.8896,126098.4,1.438059,3274.0,5.0,169776.614956,1.381427e+06,6.576875e+06
40352388,3.0,3.0,3.0,5.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,5.0,1.00,599484.250319,1.500000e-07,50000.0,1000.000000,10000000.0,1000.0,0.01,16.0,1.0,2.0,4.0,4.0,1.0,1.0,1.0,0.75,0.25,0.333333,0.5,2.5,0.25,3.591535e+06,17.7440,0.0,4.203420e+06,58490.621168,137897.2,13.8192,8.3600,121961.0,1.438892,3213.6,5.0,143564.043990,1.695668e+06,6.009714e+06


## 1. Understanding the statistics of responses to an infection

In [3]:
# Create additional variables
virs = np.unique(mean_df[['I_0','d_I','K_IE','b_I','K_EH','N_0']].to_numpy(), axis = 0)

mean_df['antigenicity_over_harm'] = antigenicity_over_harm(mean_df)
mean_df['T_pE_clear'] = mean_df['T_pE_max'] - mean_df['T_max_pI']
mean_df['max_eM_fold'] = np.log10(1 + mean_df['max_eM']/mean_df['N_0'])
mean_df['max_cM_fold'] = np.log10(1 + mean_df['max_cM']/mean_df['N_0'])
mean_df['stim_pI'] = np.log10(1 + (mean_df['p_load']/mean_df['K_IE'])/sim_duration)
mean_df['stim_pHI'] = np.log10(1 + (mean_df['int_pHI']/mean_df['K_EH'])/sim_duration)
mean_df['stim_pHE'] = np.log10(1 + (mean_df['int_pHE']/mean_df['K_EH'])/sim_duration)
mean_df['max_pE_fold'] = np.log10(1 + mean_df['max_pE']/mean_df['N_0'])

# identify Biologically evidenced networks
keep_vars = ['harm_pI', 'harm_pS', 'max_eM_fold', 'max_cM_fold', 'T_eM_min',
             'T_min_pI', 'T_max_pI', 'T_pE_start', 'T_pE_clear',
             'stim_pI', 'stim_pHI', 'stim_pHE',
             'min_pS', 'antigenicity_over_harm']

In [4]:
# save data sets
infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_0

for l, (I_0, d_I, K_IE, b_I, K_EH, N_0) in enumerate(tqdm(virs)):
    data = mean_df.loc[(mean_df["d_I"] == d_I)*(mean_df["K_IE"] == K_IE)*(mean_df["b_I"] == b_I)*(mean_df["K_EH"] == K_EH)*(mean_df["N_0"] == N_0)*(mean_df["I_0"] == I_0), 
    ['b_I','d_I', 'K_IE', 'I_0','S_0', 'N_0', 'd_S', 'K_EH'] + Na_reg + NE_reg + EM_reg + EE_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_IE = K_IE, d_I = d_I, b_I = b_I,
                                   infection_model = "cancer" if b_I >= b_C else "acute")
    no_eff_stats = no_eff_data[l]["summary_stats"]

    data.loc[:,"harm_pI_noprotection"] = no_eff_stats[3]/(b_S*sim_duration)
    data.loc[:,"peff_clearance"] = (no_eff_stats[3] - data['harm_pI'].to_numpy())/(b_S*sim_duration)
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/(b_S*sim_duration)

    infection_scenarios.append(data)

# stack datasets
clustered_mean_df = pd.concat(infection_scenarios)
clustered_mean_df.to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(infection_scenarios, f)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 130/130 [04:54<00:00,  2.27s/it]


In [5]:
del clustered_mean_df, infection_scenarios

In [6]:
# %run ./fit-observables.ipynb